In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import glob, os
import matplotlib.pyplot as plt
# import cartopy.crs as ccrs
import datetime
from multiprocessing import Pool
import importlib
from importlib import reload

import functions.outlier_removal_single_float as outlier_removal
import functions.float_download_sprof_meta as fl_download
import functions.flag_sensor_info as fl_flags 
from tqdm import tqdm
import io
import functions.float_dataframe_utils as fl_df_utils


#### Load paths from user-generated file

In [2]:
# read in a user-created text file to point to local directories to avoid having to change this every time 
# we update code
lines=[]
with open('path_file.txt') as f:
    lines = f.readlines()
    
count = 0
for line in lines:
    count += 1
    index = line.find("=")
    #print(f'line {count}: {line}')
    #print(index)
    #print(line[0:index])
    line = line.rstrip()
    if line[0:index].find("argo")>=0:
        sprof_path=line[index+1:]
    # elif line[0:index].find("liar")>=0:
    #     liar_dir=line[index+1:]
    elif line[0:index].find("matlab")>=0:
        matlab_dir=line[index+1:]
        
# Set the path for the processed files
main_argo_dir = sprof_path + '../'
output_dir = main_argo_dir + 'processed/'
interpolated_directory = main_argo_dir + 'interpolated/'
# data_dir = 'data/'


# Check for needed directories, create if it does not exist
if not os.path.isdir(sprof_path):
    os.mkdir(sprof_path)
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)
if not os.path.isdir(interpolated_directory):
    os.mkdir(interpolated_directory)

# make a directory for saving float outlier figures if not already there
if not os.path.isdir('Figures/'):
    os.mkdir('Figures')

processed_fig_dir = output_dir + 'Processed_Figures/'
# make a directory for saving processed figures if not already there
if not os.path.isdir(processed_fig_dir):
    os.mkdir(processed_fig_dir)

local_outlier_dir = '../' + 'outlier_files/'
# make a directory for storing outlier files if not already there
if not os.path.isdir(local_outlier_dir):
    os.mkdir(local_outlier_dir)

group_outlier_dir = main_argo_dir + 'outlier_file_collection/'
# make a directory for storing outlier files if not already there
if not os.path.isdir(group_outlier_dir):
    os.mkdir(group_outlier_dir)

print('Float directory used: ' + sprof_path)

Float directory used: /Users/sethbushinsky/UHM_Ocean_BGC_Group Dropbox/Datasets/Data_Products/BGC_ARGO_GLOBAL/2025_01_24/Sprof/


In [ ]:
#### Download Sprof and meta files - don't run every time
def argo_download(sprof_path):
    #get indices of BGC floats (DOXY, PH_IN_SITU_TOTAL, NITRATE)
    # First downloads the argo_synthetic-profile_index.txt file, then looks in that file for the wmo ids that correspond to BGC floats
    # probably should add a check for Fluor / Backscatter / Rad data at some point
    wmoids_doxy, gdac_index = fl_download.argo_gdac(save_to=sprof_path,sensors='DOXY',floats=None,
                                overwrite_index=True,
                                skip_download=True)
    wmoids_ph, gdac_index = fl_download.argo_gdac(save_to=sprof_path,sensors='PH_IN_SITU_TOTAL',floats=None,
                                skip_download=True)
    wmoids_nitrate, gdac_index = fl_download.argo_gdac(save_to=sprof_path,sensors='NITRATE',floats=None,
                                skip_download=True)
    #combine wmoids to one unique list
    wmoids = np.concatenate((wmoids_doxy,wmoids_ph,wmoids_nitrate))
    wmoids_all = np.unique(wmoids)
    wmoids_all = wmoids_all[~np.isnan(wmoids_all)].tolist()
    print('WMO IDs collected, ' + str(len(wmoids_all)) + ' floats to process ')

    #re-run argo_gdac with full list and download Sprof and meta files
    wmoids_bgc, gdac_index, downloaded_filenames = fl_download.argo_gdac(save_to=sprof_path,floats=wmoids_all,
                                skip_download=False, download_meta=True)
    
#### Function for plotting flag filtering if desired


# apply selected flags to data



In [ ]:
### Download Core Argo data - only run if you specifically want non-bgc floats

def argo_core_download(prof_path):
    wmoids_core, gdac_index = fl_download.argo_gdac_CORE_floats(lat_range=[-90, 90],lon_range=None,start_date=None,end_date=None,sensors=None,floats=None,
                overwrite_index=False,overwrite_profiles=False,skip_download=True,
                download_individual_profs=False,download_meta=False,save_to=None,verbose=True)

    wmoids_all = wmoids_core[~np.isnan(wmoids_core)].tolist()
    wmoids_core_out, gdac_index_out, downloaded_filenames = fl_download.argo_gdac_CORE_floats(floats=wmoids_all,
                               skip_download=False, download_meta=False,save_to=prof_path, verbose=True, 
                               download_individual_profs=False)

### Check existing Sprof files, Query whether to download new files

In [ ]:

# list number of argo files that already exist on the path
# get list of argo files
# can put in a flag here to select either sprof_path or SOCCOM_path (doesn't exist yet) to load SOCCOM data instead
argolist = []
for file in os.listdir(sprof_path):
    if file.endswith('Sprof.nc'):
        argolist.append(file)
print('Sprof_path is: ' + sprof_path)


query = "Do you want to download new data? (Y/N): "
response = input(query)

# Check the user's response and act accordingly
if response.upper() == "Y":
    print("Downloading new Sprof files.")
    argo_download(sprof_path)
elif response.upper() == "N":
    print("Skipping file download, continuing float processing")
    # Add your code here to perform actions when the user selects 'N'.
else:
    print("Invalid response. Please enter 'Y' for Yes or 'N' for No.")

print(' ')


### Apply flags, load meta data, save as temporary "Sprof_filtered" files

In [ ]:
limited_test = 0 # 0- run all floats; 1- only run the number in "num_to_test"

if limited_test==1:
    num_to_test = 5
    print('Limited test run, Only processing the last ' + str(num_to_test) + ' floats')

rerun_flags = 0 #, 0 or 1 if set to 0, will only rerun files that do not have a filtered file created

flags_to_remove = [3,4]
# WOCE quality control flags:
# 0 - No QC
# 1 - Good
# 2 - Probably good
# 3 - Bad data, potentially correctable
# 4 - Bad data
# 5 - Value changed
# 6 - Not used
# 7 - Not used
# 8 - Estimated value
# 9 - Missing value (no data)

# get list of argo files 
argolist = []
for file in os.listdir(sprof_path):
    if file.endswith('Sprof.nc'):
        argolist.append(file)

if rerun_flags ==1:
    list_to_run = argolist
else:
    filtered_list = []
    for file in os.listdir(output_dir):
        if file.endswith('filtered.nc'):
            filtered_list.append(file)

    #check which files haven't been run yet
    list_to_run = []
    match = 0
    for file in argolist:
        for filtered in filtered_list:
            #check file against every filtered file name
            if file[0:7]==filtered[0:7]:
                match = 1 #if a match is found, toggle match to 1, break loop
                break

        if match==0: #only append if file is not found in filter list        
            list_to_run.append(file) # 
        else:
            match=0

if limited_test==1:
    temp_list = list_to_run.copy()
    list_to_run = []
    list_to_run = temp_list[0:num_to_test]
    print('Limited run, Only processing the first ' + str(num_to_test) + ' floats')
else:
    if rerun_flags ==0:
        print('Number of floats with no filtered file created: ' + str(len(list_to_run)))

    else:
        print('Processing all floats, n= ' + str(len(list_to_run)))

In [ ]:
# if you want to re-run specific floats, you can uncomment/edit these lines accordingly
list_to_run = ['1900943_Sprof.nc']
# list_to_run.append('5905505_Sprof.nc')
# list_to_run.append('5906636_Sprof.nc')
# list_to_run.append('5906624_Sprof.nc')
# list_to_run = list_to_run[0:1]

In [ ]:
# flag filtering, parallel version

print('Setting flags ' + str(flags_to_remove) + ' to nan in newly created variables ("[VAR]_ADJUSTED_BGCArgoPlus"), adding sensor info and DOXY air/not air info')
print('Saving temporary "filtered" files in ' + output_dir)

from importlib import reload
reload(fl_flags)

num_processes = 6
verbose = False

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        filtered_args = [(sprof_path, file, flags_to_remove, output_dir, verbose) for n, file in enumerate(list_to_run)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(fl_flags.sensor_flag_wrapper, filtered_args)
    


### Run flag and mode filtered products only for comparison to Secondary QCed data

In [ ]:
rerun_flags = 0

flags_to_remove = [3,4]
# 0 - No QC
# 1 - Good
# 2 - Probably good
# 3 - Bad data, potentially correctable
# 4 - Bad data
# 5 - Value changed
# 6 - Not used
# 7 - Not used
# 8 - Estimated value
# 9 - Missing value (no data)

# get list of argo files 
argolist = []
for file in os.listdir(sprof_path):
    if file.endswith('Sprof.nc'):
        argolist.append(file)

if rerun_flags ==1:
    list_to_run = argolist
else:
    filtered_list = []
    for file in os.listdir(output_dir):
        if file.endswith('Sprof_flags_mode_only.nc'):
            filtered_list.append(file)

    #check which files haven't been run yet
    list_to_run = []
    match = 0
    for file in argolist:
        for filtered in filtered_list:
            #check file against every filtered file name
            if file[0:7]==filtered[0:7]:
                match = 1 #if a match is found, toggle match to 1, break loop
                break

        if match==0: #only append if file is not found in filter list        
            list_to_run.append(file) # 
        else:
            match=0

if rerun_flags ==1:
    print('Processing all floats, n= ' + str(len(list_to_run)))
else:
    print('Number of floats with no flag_mode file created: ' + str(len(list_to_run)))


In [ ]:
# if you want to re-run specific floats, you can uncomment/edit these lines accordingly
list_to_run = ['2901551_Sprof.nc']

In [ ]:
# flag, mode filtering ONLY, parallel version

print('Setting flags ' + str(flags_to_remove) + ' to nan in newly created variables')
print('Saving temporary "flag_mode" files in ' + output_dir)

from importlib import reload
reload(fl_flags)

num_processes = 6

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        filtered_args = [(sprof_path, file, flags_to_remove, output_dir, False) for n, file in enumerate(list_to_run)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(fl_flags.flag_wrapper_only, filtered_args)
    


### Outlier detection 

#### Initial run to determine which cells require outlier detection - only for oxygen, nitrate, pH floats 
Only need to run this once

In [ ]:
## Run this once to avoid doing outlier detection when you don't need to bc of missing T/S/BGC

import functions.valid_bgc_check as valid_bgc_check
importlib.reload(valid_bgc_check)

filtered_list = []
for file in os.listdir(output_dir):
    if file.endswith('Sprof_filtered.nc'):
        filtered_list.append(file)
filtered_list = np.sort(filtered_list)
print(len(filtered_list))

bgc_vars = ['NITRATE_ADJUSTED_BGCArgoPlus',
                'DOXY_ADJUSTED_BGCArgoPlus',
                'PH_IN_SITU_TOTAL_ADJUSTED_BGCArgoPlus']

# flag filtering, parallel version

print('Checking filtered files for valid bgc, saving out outlier files with XXX as researcher if not')
print('Note, this only should need to be run once')


num_processes = 7
# verbose = True

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        filtered_args = [(output_dir, file, bgc_vars, local_outlier_dir) for n, file in enumerate(filtered_list)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(valid_bgc_check.check_for_valid_bgc, filtered_args)
    




#### Optional: find pH floats only. Only need to run this if you are focusing on pH for outlier detection, or load the list below

In [ ]:
argolist = []
for file in os.listdir(output_dir):
    if file.endswith('Sprof_filtered.nc'):
        argolist.append(file)

print(len(argolist))
ph_var = 'PH_IN_SITU_TOTAL_ADJUSTED_RO'
# find floats that have valid pH
float_pH_list = []
for n, file in enumerate(argolist):
    argo_n = xr.open_dataset(output_dir + file)
    if ph_var in argo_n.keys():
        if np.sum(~np.isnan(argo_n[ph_var]))>0:
            float_pH_list.append(file)
float_pH_list = np.sort(float_pH_list)   
print(len(float_pH_list))
np.save('plotting_scripts/pH_file_list.npy', float_pH_list )


#### Outlier Removal Cell - will load each float - if outliers are removed, will save identified points in a .csv file
If no outliers are identified, creates and saves an empty .csv file to indicate that the float has been checked for outliers 


In [3]:


importlib.reload(outlier_removal)    

no_outlier_file = False # if True, returns a list of floats with no outlier files, can do this either in your own directory or in the group directory
remove_researcher_files = True # excludes floats that you have already looked at
pH_floats_in_particular = True # if true, gives a list of pH floats the researcher has not looked at yet. Must run the preceding block of code first
only_one_outlier_researcher = False  # if true, only gives a list of floats that only one person has looked at so far
only_seth = False # for identifying floats that only Seth has looked at. 

if pH_floats_in_particular:
    try:
        float_pH_list = np.load('pH_file_list.npy')
        print(len(float_pH_list))

    except:
        print('pH float list not found at : "pH_file_list.npy", check path or run preceding block to generate')

researcher = input('ENTER initials to be appended to outlier detection file, ex: SMB. Helps keep track of which floats have been assessed by multiple people.')
    
# get list of filtered files saved out
filtered_list = []
for file in os.listdir(output_dir):
    if file.endswith('Sprof_filtered.nc'):
        filtered_list.append(file)
filtered_list = np.sort(filtered_list)
# run outlier detection code or read in .csv file of bad data
# loop through all "filtered" files in argolist

outlier_list = []
for file in os.listdir(group_outlier_dir):
    if file.endswith('.csv'):
        outlier_list.append(file)
outlier_list = np.sort(outlier_list)

outlier_list = np.sort(outlier_list)
outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)
outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)

outlier_df = outlier_df.where(outlier_df['person']!='automaticoutliers').dropna(how='all')

if only_one_outlier_researcher:
    outlier_df_wmo_grouped = outlier_df.groupby('wmo')
    # print('here')
    filtered_list = []
    # different than some of the other options, here we are putting together a list of what still needs to be done, not what has already been done
    for outlier_n in outlier_df_wmo_grouped: # loop through every wmo that has been evaluated
        if np.sum(outlier_n[1]['person']=='XXX')>0: #if marked as bad, skips adding this to the list 
            continue
        # make sure you aren't double counting someone who has used their initials as capital and lowercase on the same float
        people = []
        for person in outlier_n[1]['person'].values:
            people.append(person.lower())
        if len(np.unique(people))<2: # if only one person has evaluated the file, 
            if only_seth: # if this has only been reviewed by Seth, then add it to the list
                trimmed = outlier_n[1].where(np.logical_or(outlier_n[1]['person']=='smb', 
                                 outlier_n[1]['person']=='SMB')).dropna(how='all')
            elif remove_researcher_files:    # or if that person is not the researcher:
                trimmed = outlier_n[1].where(outlier_n[1]['person']!=researcher.lower()).dropna(how='all')
                trimmed = trimmed.where(outlier_n[1]['person']!=researcher.upper()).dropna(how='all')
            else:
                trimmed = outlier_n[1]

            # then add the wmo to the filtered list
            if len(trimmed)>0:
                if pH_floats_in_particular: # check if this file is in the float pH list and only add if it is
                    if str(int(outlier_n[0]))+'_Sprof_filtered.nc' in float_pH_list:
                        filtered_list.append(str(int(outlier_n[0]))+'_Sprof_filtered.nc')
                else:
                    filtered_list.append(str(int(outlier_n[0]))+'_Sprof_filtered.nc')

    filtered_list = np.sort(filtered_list)
    
elif remove_researcher_files:
    mask1 = outlier_df['person']==researcher.lower()
    mask2 = outlier_df['person']==researcher.upper()
    mask3 =  outlier_df['person']=='XXX'
    outlier_single_person = outlier_df.where(mask1 | mask2 | mask3).dropna(how='all') # selecting all of the outliers that have been done by the researcher entered at the prompt
    outliers_unique_wmo_list = np.unique(outlier_single_person['wmo'])
   
    evaluated_list = []
    for wmo in outliers_unique_wmo_list:
        evaluated_list.append(str(int(wmo))+'_Sprof_filtered.nc')
    if pH_floats_in_particular:
        un_evaluated_list = set(float_pH_list) - set(evaluated_list)
    else:
        un_evaluated_list = set(filtered_list) - set(evaluated_list)
    filtered_list = list(un_evaluated_list)
    filtered_list = np.sort(filtered_list)

elif no_outlier_file:
    
    outliers_unique_wmo_list = np.unique(outlier_df['wmo'])

    evaluated_list = []
    for wmo in outliers_unique_wmo_list:
        evaluated_list.append(str(int(wmo))+'_Sprof_filtered.nc')

    un_evaluated_list = set(filtered_list) - set(evaluated_list)
    filtered_list = list(un_evaluated_list)
    filtered_list = np.sort(filtered_list)
    
print(len(filtered_list))

#  calling outlier_removal on a per float basis. 

idx = input('What index do you want to start at (0 to ' + str(len(filtered_list)-1) + ')? You can also enter a float WMO number to start at a specific float.')

if int(idx) > 1000000: # if gave a float number
    idx = [int(i) for i, x in enumerate(filtered_list) if idx in x][0]

for float_index in range(int(idx), len(filtered_list)):
    print('Index: ' + str(float_index) + ', WMO: ' + filtered_list[float_index])
    success = outlier_removal.pre_load_data(output_dir, file=filtered_list[float_index], researcher=researcher, port_num=8057, verbose=False)   
    next_query = input('When ready to go to the next float, hit any key (then press enter), or type "exit" to leave this loop')
    
    # Always saves an outlier file now, even if exiting.
    # Check if identified outliers, and if not save empty csv file
    savename = '../outlier_files/outliers_' + filtered_list[float_index][0:-3] + '_' + researcher.lower()

    outlier_csv_list = os.listdir('../outlier_files/')
    wmo_str = filtered_list[float_index][0:-3].split('_')[0]
    
    if any(wmo_str in s for s in outlier_csv_list) == False:
        current_time = datetime.datetime.now()
        current_time_str = str(current_time.year) + '_' + str(current_time.month) + '_' + str(current_time.day) + '_' + str(current_time.hour)
            
        if success==False:
            empty_df = pd.DataFrame(columns = ["Status"], data=['Failed to load'])
        else:
            
            empty_df = pd.DataFrame(columns = ["Float Number","Variable","N_PROF","Date","N_LEVELS",'Pressure (dbar)','Deletion reason'])
        empty_df.to_csv(savename + '_' + current_time_str + '.csv', mode='w', index=False, header=True)

    if next_query.lower()=='exit': # exits script
        break

438
29
Index: 0, WMO: 5905993_Sprof_filtered.nc
5905993_Sprof_filtered.nc
['TEMP_ADJUSTED_BGCArgoPlus', 'PSAL_ADJUSTED_BGCArgoPlus', 'NITRATE_ADJUSTED_BGCArgoPlus', 'DOXY_ADJUSTED_BGCArgoPlus', 'PH_IN_SITU_TOTAL_ADJUSTED_BGCArgoPlus']
Dash app running on http://127.0.0.1:8057/


### Count how many outlier detection files there are, etc. 


In [ ]:
# get list of filtered files
filtered_list = []
for file in os.listdir(output_dir):
    if file.endswith('Sprof_filtered.nc'):
        filtered_list.append(file)
filtered_list = np.sort(filtered_list)

print('Number of "filtered" floats = ' + str(len(filtered_list)))
# number of outlier files:

outlier_list = []
for file in os.listdir(group_outlier_dir):
    if file.endswith('.csv'):
        outlier_list.append(file)
outlier_list = np.sort(outlier_list)

# print(len(outlier_list))

import io 

outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)
outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)

# remove all "bottomoxygen" files from the outlier_df:
outlier_df = outlier_df.where(outlier_df['person']!='bottomoxygen').dropna(how='all')
# remove all "automaticoutliers" files from the outlier_df:
outlier_df = outlier_df.where(outlier_df['person']!='automaticoutliers').dropna(how='all')

XXX_files = outlier_df.where(outlier_df['person']=='XXX').dropna(how='all')
print(str(len(np.unique(XXX_files['wmo']))) + ' files have missing/non-valid T, S, or BGC data')

# non_XXX_files = outlier_df.where(outlier_df['person']!='XXX').dropna(how='all')

outliers_unique_wmo_list = np.unique(outlier_df['wmo'])
num_outliers_unique_wmo = len(outliers_unique_wmo_list)
num_valid_floats = len(filtered_list) - len(np.unique(XXX_files['wmo']))
print('Of the remaining ' +  str(num_valid_floats) + ' floats: ') #, ' + \
    #   str(num_outliers_unique_wmo) + ' floats have been reviewed, or ' + str(np.round(num_outliers_unique_wmo/len(filtered_list)*100,1)) + '%')
num_reviewers = np.zeros((len(outliers_unique_wmo_list)))
for csv_n in range(len(outliers_unique_wmo_list)):
    # print(outliers_unique_wmo_list[csv_n])
    wmo_index = outlier_df['wmo']==outliers_unique_wmo_list[csv_n]
    reviewer_initials_all = []
    for i in np.arange(len(outlier_df['person'][wmo_index])):
        # print(i)
        reviewer_initials_all.append(outlier_df['person'][wmo_index].values[i].lower())
    reviewer_initials = np.unique(reviewer_initials_all)
    # only count if the file has valid data
    if reviewer_initials.__contains__('xxx'):
        continue
    # if reviewer_initials.__contains__('n'):
    #     break
    num_reviewers[csv_n] = len(reviewer_initials)

total_percent = 0
total_percent_greater_than_two = 0
for num_n in np.arange(1, max(num_reviewers)+1):
    print('  ' + str(np.sum(num_reviewers==num_n)) + ' (' + str(np.round(np.sum(num_reviewers==num_n)/num_valid_floats*100,1)) + '%) have been reviewed by exactly ' + str(num_n) + ' reviewers')
    # print('Percent floats with ' + str(num_n) + ' reviewers = ' + str(np.round(np.sum(num_reviewers==num_n)/len(filtered_list)*100,1)) + '%, n= ' + str(np.sum(num_reviewers==num_n)))
    total_percent = total_percent + np.sum(num_reviewers==num_n)/num_valid_floats*100
    if num_n>1:
        total_percent_greater_than_two = total_percent_greater_than_two + np.sum(num_reviewers==num_n)/num_valid_floats*100

print(str(np.round(total_percent,1)) + '% of valid floats have been reviewed by at least one person')
print(str(np.round(total_percent_greater_than_two,1)) + '% of valid floats have been reviewed by at least two people')

In [ ]:
# Count total number of profiles that need to be examined
n_prof_count = 0
for idx, file in enumerate(tqdm(filtered_list)):
    if np.sum(XXX_files['wmo']==int(filtered_list[idx][0:7]))>0:
        continue
        # print(file)
        # break
    else:
        argo_n = xr.open_dataset(output_dir + file)
        n_prof = len(argo_n['N_PROF'])
        n_prof_count = n_prof_count + n_prof
        # break
print('Total number of profiles that need to be secondary QCed: ' + str(n_prof_count))

### Remove outliers and calculate derived variables

In [ ]:

matlab_code_dir = 'matlab_code_for_processing/'
if not os.path.isdir(matlab_code_dir):
    print('Missing folder containing Matlab code needed for processing. Derived variable calculation step will fail.')
    
    
# read in list of filtered files to process

rerun_flags = 1 # if set to 1, will only run files that do not have a processed file created
# get list of argo files 
argolist = []
for file in os.listdir(output_dir):
    if file.endswith('filtered.nc'):
        argolist.append(file)

if rerun_flags ==0:
    list_to_run = argolist
else:
    filtered_list = []
    for file in os.listdir(output_dir):
        if file.endswith('BGCArgoPlus_full.nc'):
            filtered_list.append(file)

    #check which files haven't been run yet
    list_to_run = []
    match = 0
    for file in argolist:
        for filtered in filtered_list:
            #check file against every filtered file name
            if file[0:7]==filtered[0:7]:
                match = 1 #if a match is found, toggle match to 1, break loop
                break

        if match==0: #only append if file is not found in filter list        
            list_to_run.append(file) # 
        else:
            match=0

list_to_run = np.sort(list_to_run)   


print('total filtered files: ' + str(len(argolist)))

if rerun_flags==0:
    print('number of files set to process: ' + str(len(list_to_run)))
else:
    print('number without processed files already created: ' + str(len(list_to_run)))

filtered_list = []
for file in os.listdir(output_dir):
    if file.endswith('Sprof_filtered.nc'):
        filtered_list.append(file)
filtered_list = np.sort(filtered_list)

print(len(filtered_list))
# go through each filtered file, check for the presence of outlier files

outlier_files = os.listdir(group_outlier_dir)
outlier_list = []
for file in os.listdir(group_outlier_dir):
    if file.endswith('.csv'):
        outlier_list.append(file)
outlier_list = np.sort(outlier_list)

outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)
outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)
# remove all outlier files w/ bottom oxygen
outlier_df = outlier_df.where(outlier_df['person']!='bottomoxygen').dropna(how='all')
# remove all outlier files w/ bottom oxygen
outlier_df = outlier_df.where(outlier_df['person']!='automaticoutliers').dropna(how='all')
# remove all outlier files w/ XXX as the name
outlier_df = outlier_df.where(outlier_df['person']!='XXX').dropna(how='all')


In [ ]:
# optional for testing
# list_to_run = ['6901898_Sprof_filtered.nc', '1901379_Sprof_filtered.nc', '5905501_Sprof_filtered.nc']
# list_to_run = ['4903489_Sprof_filtered.nc']
list_to_run = list_to_run[0:200]
# list_to_run

In [ ]:
# Run processing in parallel
import functions.derived_functions_matlab as fl_derived
reload(fl_derived)


data_type_to_process = '_ADJUSTED_BGCArgoPlus' # can run derived parameters on different levels of QC, for ex. 'PRES', 'PRES_ADJUSTED', 'PRES_ADJUSTED_RO' 

print('Using: ' + data_type_to_process + ' for derived parameters')
num_processes = 7
verbose = False

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        derived_args = [(output_dir, file, matlab_code_dir, data_type_to_process, outlier_list, outlier_df, verbose) for n, file in enumerate(list_to_run)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(fl_derived.calculate_derived_parameters, derived_args)
    


#### Run processing for files with only flags/delayed mode filters

In [ ]:
# optional load of list instead of cell above - only needed if you want to focus on pH floats
float_pH_list = np.load('plotting_scripts/ph_file_list.npy')
print(len(float_pH_list))
list_to_run = float_pH_list

In [ ]:

rerun_flags = 1 # if set to 1, will only run files that do not have a processed file created
run_pH_only = False # if set to 1, will only run pH floats 

matlab_code_dir = 'matlab_code_for_processing/'
if not os.path.isdir(matlab_code_dir):
    print('Missing folder containing Matlab code needed for processing. Derived variable calculation step will fail.')


# get list of argo files 
argolist = []
for file in os.listdir(output_dir):
    if file.endswith('_Sprof_flags_mode_only.nc'):
        argolist.append(file)

if rerun_flags ==0:
    if run_pH_only:
        #check which files haven't been run yet
        list_to_run = []
        # match = 0
        for file in argolist:
            for filtered in float_pH_list:
                #check file against every filtered file name
                if file[0:7]==filtered[0:7]:
                    # print(file)
                    list_to_run.append(file)
                    break
    else:
        list_to_run = argolist
else:
    filtered_list = []
    for file in os.listdir(output_dir):
        if file.endswith('Sprof_BGCArgoPlus_flags_mode_only.nc'):
            filtered_list.append(file)

    #check which files haven't been run yet
    list_to_run = []
    match = 0
    if run_pH_only:
        list_to_check = float_pH_list
    else:
        list_to_check = argolist
    for file in list_to_check:
        for filtered in filtered_list:
            #check file against every filtered file name
            if file[0:7]==filtered[0:7]:
                match = 1 #if a match is found, toggle match to 1, break loop
                break

        if match==0: #only append if file is not found in filter list        
            list_to_run.append(file) # 
        else:
            match=0

list_to_run = np.sort(list_to_run)   

print('total flag and mode-only filtered files: ' + str(len(argolist)))

if rerun_flags==0:
    print('number of files set to process: ' + str(len(list_to_run)))
else:
    print('number without processed files already created: ' + str(len(list_to_run)))



In [ ]:
list_to_run = ['2901551_Sprof_flags_mode_only.nc']

In [ ]:
# Run processing in parallel
import functions.derived_functions_matlab as fl_derived

reload(fl_derived)
verbose = True


data_type_to_process = '_ADJUSTED_BGCArgoPlus' # can run derived parameters on different levels of QC, for ex. 'PRES', 'PRES_ADJUSTED', 'PRES_ADJUSTED_RO' 

num_processes = 6

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        derived_args = [(output_dir, file, matlab_code_dir, data_type_to_process, verbose) for n, file in enumerate(list_to_run)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(fl_derived.calculate_carbonate_parameters_only, derived_args)


#### Testing cells:

## OLD

### Plotting of float data: Original, Adjusted, BGCArgoPlus

In [ ]:
# first plot: map of float location

# second plots: depth section plots for each variable:
rerun_flags = 0
# var = 'DOXY'

# get a list of all Sprof files, will go through them all
file_string = 'Sprof.nc'
# get list of filtered files saved out
processed_list = []
for file in os.listdir(sprof_path):
    if file.endswith(file_string):
        processed_list.append(file)


data_type_to_plot = '_ADJUSTED_BGCArgoPlus' # can run derived parameters on different levels of QC, for ex. 'PRES', 'PRES_ADJUSTED', 'PRES_ADJUSTED_RO' 

print('Using: ' + data_type_to_plot + ' for derived parameters')


if rerun_flags ==0:
    plot_dir_list = next(os.walk(processed_fig_dir))[1]


    #check which files haven't been run yet
    list_to_run = []
    match = 0
    for file in processed_list:
        for plot_dir in plot_dir_list:
            #check file against every filtered file name
            if file[0:7]==plot_dir[0:7]:
                match = 1 #if a match is found, toggle match to 1, break loop
                break

        if match==0: #only append if file is not found in filter list        
            list_to_run.append(file) # 
        else:
            match=0
    processed_list = list_to_run

print(len(processed_list))

outlier_files = os.listdir(group_outlier_dir)
outlier_list = []
for file in os.listdir(group_outlier_dir):
    if file.endswith('.csv'):
        outlier_list.append(file)
outlier_list = np.sort(outlier_list)

outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)
outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)
# remove all outlier files w/ bottom oxygen
outlier_df = outlier_df.where(outlier_df['person']!='bottomoxygen').dropna(how='all')
# remove all outlier files w/ bottom oxygen
# outlier_df = outlier_df.where(outlier_df['person']!='automaticoutliers').dropna(how='all')
# remove all outlier files w/ XXX as the name
outlier_df = outlier_df.where(outlier_df['person']!='XXX').dropna(how='all')

In [ ]:
# processed_list = processed_list[0:100]
# processed_list.append('7902223_Sprof.nc')
# processed_list.append('7901134_Sprof.nc')
# processed_list.append('5906765_Sprof.nc')
# processed_list.append('5905380_Sprof.nc')
processed_list = ['1902371_Sprof.nc']



In [ ]:
float_pH_list = ['2901550_Sprof.nc']

In [ ]:
num_processes = 1
import plotting_scripts.float_summary_plots as summary_plots
from importlib import reload
reload(summary_plots)

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        plot_args = [(sprof_path, output_dir, file, data_type_to_plot, outlier_list, outlier_df) for n, file in enumerate(processed_list)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(summary_plots.float_plots, plot_args)
    

In [ ]:
palette = list(plt.cm.tab10.colors)   # list of RGB tuples; use tab20, etc. if you need more
reason_colors = {'profile': palette[0], 
                   'Surface Removal': palette[1],
                   'Density Inversion': palette[2],
                   'Bottom_Oxygen_Check': palette[3],
                   'Time series': palette[4],} 

In [ ]:
reason = 'asd'
if reason in reason_colors.keys():
    color = reason_colors[reason]
else:
    color = 'k' #default to black if reason not in list
color

In [ ]:
### Testing ###
import plotting_scripts.float_summary_plots as summary_plots
from importlib import reload
reload(summary_plots)
data_type_to_plot = '_ADJUSTED_RO' # can run derived parameters on different levels of QC, for ex. 'PRES', 'PRES_ADJUSTED', 'PRES_ADJUSTED_RO' 


file = '5906220_Sprof.nc' # processed_list[0]
summary_plots.float_plots(sprof_path=sprof_path, output_dir=output_dir, file=file, data_type_to_plot=data_type_to_plot)

In [ ]:
processed_file = '5906220_Sprof_processed.nc' # processed_list[0]
file = processed_file
file_parts = file.split('_')
wmo = file_parts[0]
color_map = 'plasma_r'

argo_n = xr.open_dataset(output_dir + processed_file)
var_base = 'PSAL'
fig = plt.figure(figsize=(14,10))

# print('here')
list_var = list(argo_n.keys())

argo_n['decimal_year'] = (['N_PROF'],np.empty(argo_n.PRES_ADJUSTED.shape[0])) #nprof 
argo_n.decimal_year[:] = np.nan
date_time = pd.to_datetime(argo_n.JULD.values)
year = date_time.year
decimal_year = year + (date_time.day_of_year - 1) / 365.25
argo_n.decimal_year[:] = decimal_year
# var_types = {'_ADJUSTED_RO'}
var_types = {data_type_to_plot}
for idx, var_type in enumerate(var_types):
    data_exists=False

    ax = fig.add_subplot(2,3,idx+5)
    var_plot = var_base + var_type
    pres_name = 'PRES'
    var_min = np.nanmin(argo_n[var_plot])
    var_max = np.nanmax(argo_n[var_plot])     
    for p in range(0, len(argo_n.N_PROF)):
        # p_p = argo_n[pres_name][p,np.logical_and(~np.isnan(argo_n[var_plot][p,:]), ~np.isnan(argo_n[pres_name][p,:]))].values
        # t_p = np.array([argo_n.decimal_year[p].item(), argo_n.decimal_year[p].values + np.nanmedian(np.diff(argo_n.decimal_year))]) # try padding with median difference between profile times instead of next profile in case of large gaps
        p_p = argo_n[pres_name][p, ~np.isnan(argo_n[pres_name][p,:])].values
        t_p = np.array([argo_n.decimal_year[p].item(), argo_n.decimal_year[p].values + np.nanmedian(np.diff(argo_n.decimal_year))]) # try padding with median difference between profile times instead of next profile in case of large gaps

        # t_p = argo_n.decimal_year[p:p+2].values
        if t_p.size==1:
            t_p = np.tile(t_p, (2,1))
            t_p[1] = t_p[1] + (argo_n.decimal_year[p] - argo_n.decimal_year[p-1]).values
        elif np.isnan(t_p).any(): # if any values in t_p are nans
            if np.isnan(t_p).all(): # if all are nans, continue
                continue
            elif np.isnan(t_p[0]):
                t_p[0] = t_p[1] - 10/365
            else:
                t_p[1] = t_p[0] + 10/365
        xl,yl = np.meshgrid(t_p, p_p)

        c = argo_n[var_plot][p,~np.isnan(argo_n[pres_name][p,:])].values

        # c = argo_n[var_plot][p,np.logical_and(~np.isnan(argo_n[var_plot][p,:]), ~np.isnan(argo_n[pres_name][p,:]))].values
        if c.size==0:
            continue
        c = np.tile(c, (2,1))
        c = c.T
        # try:
        pc = plt.pcolormesh(xl, yl, c[0:-1,0:-1], cmap=color_map, shading='flat', vmin=var_min, vmax=var_max)
        data_exists = True
        # except:
            # print(file + ' ' + var_plot + ' failed to plot')
            
    if data_exists:
        plt.colorbar(pc)
            # plt.clim(c_limit)
        plt.ylim(argo_n[pres_name].max().values, 0)
        plt.xlim(argo_n['decimal_year'][0], argo_n['decimal_year'][-1])
    plt.title(var_plot)
    xtick_pos = ax.get_xticks()
    # plot outliers removed:
    ax = fig.add_subplot(2,3,idx+6)

    # Keep track of which reasons have been added to legend
    legend_reasons = set()
    
    # read in each outlier file
    for file_n in matched_file_list: 
        if verbose:
            print('Looking at contents of outlier file ' + file_n)
        # file_n = matched_file_list[o]
        # print(file_n)
        with open(group_outlier_dir+file_n) as csvfile:
            df_out = pd.read_csv(csvfile)

            # var = df_out['Variable'].values
            # nprof = df_out['N_PROF'].values
            # nlevel = df_out['N_LEVELS'].values
            # reason = df_out['Deletion reason'].values

            # if len(nprof) > 0: 
            for i in range(0, len(df_out)): # go through each row, setting all values to nans
                if df_out['Variable'][i].startswith(var_base):
                    decimal_year = pd.to_datetime(df_out['Date'][i]).year + pd.to_datetime(df_out['Date'][i]).dayofyear/365
                    if df_out['Deletion reason'][i].lower()=='density inversion':
                        color = 'r'
                    else:
                        color='b'
                    
                    # Only add label if this reason hasn't been seen before
                    reason = df_out['Deletion reason'][i].lower()
                    if reason not in legend_reasons:
                        ax.plot(decimal_year, df_out['Pressure (dbar)'][i], 's', color=color, label=reason, markersize=6, alpha=.5)
                        legend_reasons.add(reason)
                    else:
                        ax.plot(decimal_year, df_out['Pressure (dbar)'][i], 's', color=color, markersize=6, alpha=.5)
                    print(decimal_year, df_out['Pressure (dbar)'][i], reason)

                
    plt.ylim(argo_n[pres_name].max().values, 0)
    plt.xlim(argo_n['decimal_year'][0], argo_n['decimal_year'][-1])
    
    # Only show legend if there are items to display
    if legend_reasons:
        ax.legend()
    
    ax.set_title('Outliers Removed')
    ax.set_xticks(xtick_pos)

fig.suptitle(wmo + ', ' + str(len(argo_n.N_PROF)) + ' profiles, ' + str(argo_n['JULD'][0].dt.date.values) + ' to ' + str(argo_n['JULD'][-1].dt.date.values))
plt.tight_layout()

In [ ]:
# get index of where reason equals legend_reason
for reason in legend_reasons:
    if reason in legend_reasons:
        print(reason)
        leg_index = legend_reasons.__contains__(reason).__index__()
        print(leg_index)

In [ ]:
# plot outliers removed, both automatic and manual

outlier_files = os.listdir(group_outlier_dir)
outlier_list = []
for file in os.listdir(group_outlier_dir):
    if file.endswith('.csv'):
        outlier_list.append(file)
outlier_list = np.sort(outlier_list)

outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)
outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)
# remove all outlier files w/ bottom oxygen
outlier_df = outlier_df.where(outlier_df['person']!='bottomoxygen').dropna(how='all')
# remove all outlier files w/ bottom oxygen
# outlier_df = outlier_df.where(outlier_df['person']!='automaticoutliers').dropna(how='all')
# remove all outlier files w/ XXX as the name
outlier_df = outlier_df.where(outlier_df['person']!='XXX').dropna(how='all')



verbose = True
processed_file = '5906220_Sprof_processed.nc' # processed_list[0]
# file = processed_file
# file_parts = file.split('_')
# wmo = file_parts[0]
color_map = 'plasma_r'

argo_n = xr.open_dataset(output_dir + processed_file)



wmo_n = argo_n['WMO_ID'].values


matched_df = outlier_df.where(outlier_df['wmo']==wmo_n).dropna(how='all')
if verbose:
    print('Matched df:')
    print(matched_df)
if len(matched_df)>0:
    matched_file_list = outlier_list[matched_df.index]

group_outlier_dir= output_dir+ '../outlier_file_collection/'

# read in each outlier file
for file_n in matched_file_list: 
    if verbose:
        print('Looking at contents of outlier file ' + file_n)
    # file_n = matched_file_list[o]
    # print(file_n)
    with open(group_outlier_dir+file_n) as csvfile:
        df_out = pd.read_csv(csvfile)

        # var = df_out['Variable'].values
        # nprof = df_out['N_PROF'].values
        # nlevel = df_out['N_LEVELS'].values
        # reason = df_out['Deletion reason'].values

        # if len(nprof) > 0: 
        for i in range(0, len(df_out)): # go through each row, setting all values to nans
            if df_out['Variable'][i].startswith(var_base):
                print(df_out['Date'][i], df_out['Pressure (dbar)'][i])


### Generate and Save a float DataFrame of all float data

In [ ]:
num_processes = 7
argo_path = "/Users/znachod/UHM_Ocean_BGC_Group Dropbox/Datasets/Data_Products/BGC_ARGO_GLOBAL/2025_01_24/processed/"
argolist= []
for file in os.listdir(argo_path):
    if 'Sprof_BGCArgoPlus.nc' in file:
        argolist.append(file)
# for file in os.listdir(float_dir):
#     if 'Sprof_processed' in file:
#         argolist.append(file[0:7] + '_Sprof_processed.nc')
# # argolist.sort()
argolist.sort()
# argolist_run=argolist
argolist_run=argolist

if __name__ == "__main__":
    with Pool(processes=num_processes) as pool:
    # Create a list of arguments for pool.starmap
        argo_args = [(argolist_run, file, 1, 1, 1) for file in argolist_run]
        # argo_args = [(float_dir, file, 0, 0) for file in argolist_run]

    # Use pool.starmap with the list of arguments
        float_profile_data = pool.starmap(fl_df_utils.create_float_df_2, argo_args)
        # float_profile_data = pool.starmap(intro_functions.create_float_df_2, argo_args)

In [ ]:
# Combine all output dataframes, add datetime, save to csv
float_profile_data_combined = pd.concat(float_profile_data)
float_profile_data_combined_explode = float_profile_data_combined.explode(['Pressure', 'Temperature', 'Salinity','Oxygen_Corrected', 'Oxygen_Delayed_w_Flags'])
float_profile_data_combined_explode["Datetime"] = pd.to_datetime(float_profile_data_combined_explode.Datetime)
# float_profile_data_combined_explode.to_csv(argo_path + "BGCArgoPlusAllData.csv")
# all_float_data = pd.read_csv(argo_path + "all_float_data.csv")


In [ ]:
# Read in Data if not loaded already
# Rename columns to desired names and select only needed columns
selector_d = {"Datetime": "Datetime", 'Latitude':'Latitude', 'Longitude':'Longitude', 'Pressure': 'Pressure', 'Temperature': 'Temperature', 'Salinity': 'Salinity', 'Oxygen_Corrected': 'Oxygen_Corrected', 
              'Oxygen_Delayed_w_Flags': 'Oxygen_Delayed_w_Flags'}
all_float_data = float_profile_data_combined_explode.rename(columns=selector_d)[[*selector_d.values()]]

In [ ]:
# Set Bin Edges
binx=np.linspace(-179.5, 179.5, 360)
biny=np.linspace(-89.5, 89.5, 180)
binz=[0, 5, 15, 25, 35, 45, 55, 65, 75, 85, 95, 105, 115, 125, 135, 145, 155, 165, 175, 185, 210, 230, 250, 270, 290, 310, 330, 350, 370, 390, 410, 430, 450, 470,
        525, 575, 625, 675, 725, 775, 825, 875, 925, 975, 1025, 1075, 1125, 1175, 1225, 1275, 1325, 1375, 1425, 1550, 1650, 1750, 1850, 1950, 2050]
bins=[binz, biny, binx]

targetx = np.linspace(-179, 179, 359)
targety = np.linspace(-89, 89,  179)
targetz = [2.5, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0, 140.0, 150.0, 160.0, 
            170.0, 180, 200.0, 220.0, 240.0, 260.0, 280.0, 300.0, 320.0, 340.0, 360.0, 380.0, 400.0, 420.0, 440.0, 460.0, 
            500.0, 550.0, 600.0, 650.0, 700.0, 750.0, 800.0, 850.0, 900.0, 950.0, 1000.0, 1050.0, 1100.0, 1150.0, 1200.0, 
            1250.0, 1300.0, 1350.0, 1400.0, 1500.0, 1600.0, 1700.0, 1800.0, 1900.0, 2000.0]

In [ ]:
# Time to bin
import warnings

from scipy import stats
# Set Encoding (compression) for netCDF4 output
encoding = {var: {"zlib": True, "complevel": 4} for var in [
    "Temperature", "Salinity", "Oxygen"
]}
# Loop through each month and year combination
for time, data in all_float_data.groupby([all_float_data['Datetime'].dt.year.rename('year'), 
                    all_float_data['Datetime'].dt.month.rename('month')]):
    year, month = time
    # if year == 2006 and month == 9:
    print(f"Year: {year}, Month: {month}")
        #float
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        try:
            float_data = data[data["Observation Type"] == "Float"]
            spatial_data = [float_data['Pressure'].values.astype(float), float_data.Latitude.values.astype(float), float_data.Longitude.values.astype(float)]
            to_bin = [float_data['Temperature'].values.astype(float), float_data['Salinity'].values.astype(float), float_data['Oxygen_Corrected'].values.astype(float)]
            float_statistic, bin_edges, binnumber = stats.binned_statistic_dd(sample = spatial_data, values=to_bin, bins=bins, statistic=np.nanmean)
        except ValueError as e:
            float_statistic = np.full((3, len(targetz), len(targety), len(targetx)), np.nan)
    description = 'Gridded monthly mean binned data for float oxygen profiles'
    ds = xr.Dataset(
        data_vars=dict(
            Temperature=(["z", "lat", "lon"], float_statistic[0]),
            Salinity=(["z", "lat", "lon"], float_statistic[1]),
            Oxygen=(["z", "lat", "lon"], float_statistic[2]),
        ),
        coords=dict(
            z=targetz,
            lat=targety,
            lon=targetx
        ),
        attrs=dict(
            description=description
        )
    )
    ds = ds.expand_dims(time=[pd.Timestamp(year=year, month=month, day=1)])
    ds.to_netcdf("/Users/znachod/UHM_Ocean_BGC_Group Dropbox/Datasets/Data_Products/BGC_ARGO_GLOBAL/2025_01_24/processed/gridded/" + "BGCArgoPlusGridded_" + str(year) + "_" + str(month) + ".nc", encoding=encoding)

In [ ]:
# # Try to remove outlier files where T/S was originally removed due to not being delayed mode

# filtered_list = []
# for file in os.listdir(output_dir):
#     if file.endswith('Sprof_filtered.nc'):
#         filtered_list.append(file)
# filtered_list = np.sort(filtered_list)

# outlier_list = []
# for file in os.listdir(group_outlier_dir):
#     if file.endswith('.csv'):
#         outlier_list.append(file)
# outlier_list = np.sort(outlier_list)

# # print(len(outlier_list))

# import io 

# outlier_df = pd.read_csv(io.StringIO('\n'.join(outlier_list)), delimiter='_', header=None)

# outlier_df.rename(columns={1: 'wmo', 4:'person', 5:'year', 6:'month', 7:'day', 8:'hour'}, inplace=True)
# for file in tqdm(filtered_list):
#     # file = '2901558_Sprof_filtered.nc'
#     argo_n = xr.open_dataset(output_dir + file)

#     # check if any outlier files exist for this one:
#     try:
#         argo_n["WMO_ID"] = argo_n.PLATFORM_NUMBER.values.astype(int)[0]
#     except:
#         print('Error in file ' + file)
#         continue
    
#     outlier_file_matches = outlier_df.where(outlier_df['wmo']==int(argo_n["WMO_ID"].values)).dropna(how='all')
#     if len(outlier_file_matches)==0:
#         # print('here1')
#         continue
#     # is an 'XXX' file present, indicating that this float should be skipped?
#     XXX_files = outlier_file_matches.where(outlier_file_matches['person']=='XXX').dropna(how='all')
#     if len(XXX_files)>0:
#         # print('here2')

#         continue # no need to keep going if XXX file is present 

#     non_XXX_files = outlier_file_matches.where(outlier_file_matches['person']!='XXX').dropna(how='all')

#     files_empty = 1
#     for i in range(len(non_XXX_files)):
#         csv_file = open(group_outlier_dir+outlier_list[non_XXX_files.index[i]])
#         df_out = pd.read_csv(csv_file)
#         if len(df_out)>0:
#                 files_empty=0

#     # if files_empty is still set to 0, then need to check if this is empty because T/S would have prevented its being run before:
#     if files_empty==0:
#         # print('here2.5')
#         continue
    
#     argo_n["WMO_ID"] = argo_n["WMO_ID"].astype(dtype='object')

#     pres_data = argo_n['PRES_ADJUSTED'].values
#     nprof_n = argo_n.sizes['N_PROF']


#     #Finding and removing all non-delayed mode data
#     # sometimes parameters are missing from profiles - 
#     # need to loop through all profiles and check which parameters are present
#     # assumes that mode applies to all levels of a profile
#     parameter_array = argo_n.STATION_PARAMETERS.values.astype(str)

#     for idx in range(len(parameter_array)): # loop through all profiles, idx is profile index 
#         prof_parameters = parameter_array[idx] # get parameters present for each profile 
#         # print(prof_parameters)
#         # print('here3')

#         # loop through each paramter in the profile 
#         for var in prof_parameters:
#             var_str = var.strip()
#             if var_str in ['TEMP',
#                         'PSAL']:
                
#                 # if np.logical_or(len(var_str)==0, var_str=='PRES'): # only proceed if the variable exists and is not pressure
#                 #     continue
#                 # if np.logical_or(var_str=='TEMP', var_str=='PSAL'): # don't set TEMP or PSAL to nan for non-delayed mode data
#                 #     continue
#                 if var_str + '_profile_removed_not_D' not in argo_n: # if profile removed count has not been initialized, do so here 
#                     argo_n[var_str + '_profile_removed_not_D'] = 0
#                 var_ind = [idx for idx, s in enumerate(prof_parameters) if s.strip()== var_str]
#                 # print(var_ind)

#                 # get parameter data mode values for that profile / variable
#                 var_data_mode = argo_n.PARAMETER_DATA_MODE[idx,var_ind].values
#                 # print(var_data_mode)

#                 decoded_arr = np.array([elem.decode() if isinstance(elem, bytes) else np.nan for elem in var_data_mode.flatten()])
#                 # print(decoded_arr)
#                 result = np.where(decoded_arr == 'D', False, True) # true whereever mode is not delayed
#                 # print(result)
#                 if result:
#                     # argo_n[var_str +'_ADJUSTED_RO'][idx,:] = np.nan
#                     argo_n[var_str + '_profile_removed_not_D']+=1 # count the number of profiles removed b/c mode does not equal "D"
#                     # print('here3.5')
#     # print('here4')

#     # check if all profiles were removed due to not being delayed mode
#     if np.logical_or(argo_n['TEMP_profile_removed_not_D'].values==nprof_n, argo_n['PSAL_profile_removed_not_D'].values==nprof_n): 
#             # print(file)
#             # now create a file to indicate that you need to remove these - 
#             savename = group_outlier_dir + '/outliers_' + file[0:-3] + '_DELETE'
#             empty_df = pd.DataFrame(columns = ["Status"], data=['Need to re-check this float'])
#             empty_df.to_csv(savename + '.csv', mode='w', index=False, header=True)
            


In [ ]:
# OLD ## Replace bad data with nans
def apply_outlier_detection(argo_file, outlier_files):
    reviewer_symbols = ['xr', 'ob', '+k', '^m', 'sg']

    print(argo_file[:7])
    wmo = argo_file[:7]
    # find all outlier files that match the WMO, if any
    matched_file_list = [outlier_file for outlier_file in outlier_files if np.logical_and(wmo in outlier_file, not os.path.isdir(group_outlier_dir + outlier_file))]
    print(matched_file_list)

    plot_changes = True

    # find 
    if len(matched_file_list)> 0:
        argo_n = xr.open_dataset(output_dir + argo_file)
        # create a copy to use for plotting removed points
        argo_n_orig = argo_n.copy(deep=True)

        outliers_removed = False
        # open all outlier files that match the WMO
        for o in range(0, len(matched_file_list)): 
            file_n = matched_file_list[o]
            print(file_n)
            with open(group_outlier_dir+file_n) as csvfile:
                df_out = pd.read_csv(csvfile)

                var = df_out['Variable'].values
                nprof = df_out['N_PROF'].values
                nlevel = df_out['N_LEVELS'].values
        
                if len(nprof) > 0: 
                    df_out['reviewer_initials'] = file_n.split('_')[4]
                    if 'df_all' in locals():
                        df_all = pd.concat([df_all, df_out])
                    else:
                        df_all = df_out
    
                    outliers_removed=True # sets to True if any values are being changed
                    for i in range(0, len(nprof)): # go through each row, setting all values to nans
                        # Replace the data with nans
                        if type(var[i])==str:
                            argo_n[var[i]].loc[{'N_PROF':int(nprof[i]), 'N_LEVELS':int(nlevel[i])}] = np.nan

        # if there were outliers removed and plot_changes is set to True, then plot profiles that were removed     
        if np.logical_and(outliers_removed, plot_changes):

            # make a removed profile directory for this float if one doesn't exist
            float_RO_profile_dir = group_outlier_dir + '/' + str(wmo) + '/'
            if not os.path.isdir(float_RO_profile_dir):
                os.mkdir(float_RO_profile_dir)

            outlier_profiles = np.sort(df_all['N_PROF'].unique())
            for idx_p in range(0, len(outlier_profiles)):
                if np.isnan(outlier_profiles[idx_p]):
                    continue 
                plot_filename = str(wmo) + '_profile_' +  str(outlier_profiles[idx_p])

                prof_index = df_all['N_PROF']==outlier_profiles[idx_p]
                var_all = df_all['Variable'][prof_index]
                unique_variables = np.unique(var_all).tolist()
                if 'TEMP_ADJUSTED_RO' not in unique_variables:
                    unique_variables.append('TEMP_ADJUSTED_RO')
                if 'PSAL_ADJUSTED_RO' not in unique_variables:
                    unique_variables.append('PSAL_ADJUSTED_RO')                
                unique_reviewers = np.unique(df_all['reviewer_initials'])

                fig = plt.figure(figsize=(len(unique_variables)*10,10),)

                for idx_v in range(0, len(unique_variables)):


                    # loop through all removed variables types for that profile
                    # first plot that profile, plus ones before and after:
                    ax = fig.add_subplot(1, len(unique_variables), idx_v+1)

                    # make sure we don't exceed the limits of available profiles
                    first_profile = int(outlier_profiles[idx_p]-2)
                    last_profile = int(outlier_profiles[idx_p]+3)
                    if first_profile<0:
                        first_profile=0
                    if last_profile>len(argo_n_orig['N_PROF'].values)-1:
                        last_profile = argo_n_orig['N_PROF'][-1].values

                    for prof_plot in range(first_profile,last_profile):
                        # print(prof_plot)
                        ax.plot(argo_n_orig[unique_variables[idx_v]].loc[{'N_PROF':prof_plot}].values, \
                            argo_n_orig['PRES'].loc[{'N_PROF':prof_plot}].values, label = prof_plot)
                        
                    # plt.plot(argo_n_orig[unique_variables[idx_v]].loc[{'N_PROF':outlier_profiles[idx_p]}].values, \
                    #          argo_n_orig['PRES'].loc[{'N_PROF':outlier_profiles[idx_p]}].values, label =)
                    ax.set_title(unique_variables[idx_v] + ' Profile: ' + str(outlier_profiles[idx_p]))

                    # plot removed variables:

                    df_prof_var = df_all[prof_index].where(df_all['Variable'][prof_index]==unique_variables[idx_v]) 
                    df_prof_var = df_prof_var.drop_duplicates().dropna(how='all').reset_index()

                    # add an index item for the reviewers present in the profile
                    prof_reviewers = np.unique(df_prof_var['reviewer_initials'])
                    for pr in range(0, len(prof_reviewers)):
                        reviewer_match = unique_reviewers==prof_reviewers[pr]
                        rev_index = [i for i, x in enumerate(reviewer_match) if x] # get an index for the reviewer so that you have different legend values 

                        ax.plot(np.nan, np.nan, reviewer_symbols[rev_index[0]], label=unique_reviewers[rev_index])
                    label=df_prof_var['reviewer_initials']
                    # loop through rows and plot outlier points
                    for RO_n in range(0, len(df_prof_var)):
                        reviewer_match = unique_reviewers==df_prof_var['reviewer_initials'][RO_n] 
                        rev_index = [i for i, x in enumerate(reviewer_match) if x] # get an index for the reviewer so that you have different legend values 
                        
                        # print(argo_n_orig[unique_variables[idx_v]].loc[{'N_PROF':outlier_profiles[idx_p], 'N_LEVELS':int(df_prof_var['N_LEVELS'][RO_n])}].values)
                        ax.plot(argo_n_orig[unique_variables[idx_v]].loc[{'N_PROF':int(outlier_profiles[idx_p]), 'N_LEVELS':int(df_prof_var['N_LEVELS'][RO_n])}].values, \
                            argo_n_orig['PRES'].loc[{'N_PROF':int(outlier_profiles[idx_p]), 'N_LEVELS':int(df_prof_var['N_LEVELS'][RO_n])}].values, reviewer_symbols[rev_index[0]] )
                        
                    ax.invert_yaxis()
                    ax.legend()
                plt.tight_layout()
                plt.savefig(f'{float_RO_profile_dir}{plot_filename}.png')
                plt.close(fig)
    # # Load data
    # argo_n = xr.open_dataset(output_dir + file)
    # fnum = argo_file.split('_')[0]
    
    # # Load outlier file
    # float_outlier_files = [each for each in outlier_files if fnum in each]
    
    # if len(float_outlier_files) == 0:
    #     print('No outlier file. Run outlier detection for this file:', fnum)
    #     return 

    
    # # elif len(outlier_file) > 1:
    # #     print('Multiple files for same float. Not implemented')  
    # # outlier_file = outlier_file[0]
    # # now applies all outliers listed in outlier files that match the wmo number 
    # for o in range(0, len(float_outlier_files)): 
    #     file_n = float_outlier_files[o]
    #     with open('../outlier_files/'+file_n) as csvfile:
    #         df_out = pd.read_csv(csvfile)
    #         var = df_out['Variable'].values
    #         nprof = df_out['N_PROF'].values
    #         nlevel = df_out['N_LEVELS'].values
    
    #         if len(nprof) > 1:
                
    #             # Replace the data with nans
    #             argo_n[var].loc[{'N_PROF':nprof, 'N_LEVELS':nlevel}] = np.nan

    # # save intermediate file
    print('here')
    argo_n.to_netcdf(output_dir + argo_file[:-3] + '_RO.nc')
    argo_n.close()        
    argo_n_orig.close()
    return 

In [ ]:
# Plotting summary figures for each float
from importlib import reload
reload(fl_flags)

num_processes = 18

# plot_vars_all = ['TEMP_ADJUSTED', 'PSAL_ADJUSTED', 'sigma0', 'gamma', 'DOXY_ADJUSTED', 'NITRATE_ADJUSTED', 
#                  'PH_IN_SITU_TOTAL_ADJUSTED', 'TALK_LIAR', 'DIC', 'CHLA_ADJUSTED', 'BBP700_ADJUSTED', 'CDOM_ADJUSTED',
#                  'DOWNWELLING_PAR_ADJUSTED', 'DOWN_IRRADIANCE380_ADJUSTED', 'DOWN_IRRADIANCE412_ADJUSTED', 'DOWN_IRRADIANCE490_ADJUSTED',
#                  'CP660_ADJUSTED']
# plot Sprof files
# output_dir = sprof_path # non-interpolated files
# file_string = 'Sprof.nc'

# plot processed files
output_dir = output_dir # non-interpolated files
file_string = 'Sprof_processed.nc'
# get list of filtered files saved out
processed_list = []
for file in os.listdir(output_dir):
    if file.endswith(file_string):
        processed_list.append(file)

print(len(processed_list))

In [ ]:

if __name__ == "__main__":
    
    with Pool(processes=num_processes) as pool:
        # Create a list of arguments for pool.starmap
        plot_args = [(file, output_dir, processed_fig_dir) for n, file in enumerate(processed_list)]
        
        # Use pool.starmap with the list of arguments
        pool.starmap(fl_flags.plot_processed_files, plot_args)
    


In [ ]:
# test cell for plotting
file = '6902900_Sprof_processed.nc'
# file = '4903739_Sprof.nc'

output_dir = output_dir # non-interpolated files
# output_dir = sprof_path # non-interpolated files

# # get list of filtered files saved out
# processed_list = []
# for file in os.listdir(output_dir):
#     if file.endswith('Sprof_processed.nc'):
#         processed_list.append(file)


# for n, file in enumerate(processed_list): #enumerate(temp_list): # 
argo_n = xr.open_dataset(output_dir + file)


In [ ]:

argo_n['decimal_year'] = (['N_PROF'],np.empty(argo_n.PRES_ADJUSTED.shape[0])) #nprof 
argo_n.decimal_year[:] = np.nan
date_time = pd.to_datetime(argo_n.JULD.values)
year = date_time.year
decimal_year = year + (date_time.day_of_year - 1) / 365.25
    
argo_n.decimal_year[:] = decimal_year

data_proj = ccrs.PlateCarree(central_longitude=0)
map_proj = ccrs.Robinson(central_longitude=np.nanmean(argo_n['LONGITUDE']))

list_var = list(argo_n.keys())
plot_vars = [field for field in list_var if field.endswith('_ADJUSTED')]


# list_var = list(argo_n.keys())
# plot_vars = []
# # list_var = list(argo_n.keys())
# for var in plot_vars_all:
#     if var in list_var:
#         plot_vars.append(var)

# print('variables in file:')
# print(plot_vars)
# # Then, plot all kind of fields to see if there are outliers
f = plt.figure(figsize=(40,5*len(plot_vars)))

gs = f.add_gridspec(1+len(plot_vars), 2)

ax0 = f.add_subplot(gs[0], projection=map_proj)
ax0.set_global()
ax0.coastlines()
# ax0.set_title( "%s, %s to %s"%(fname,dates[0].astype('datetime64[D]'),dates[-1].astype('datetime64[D]')) )
map = ax0.scatter(argo_n.LONGITUDE.values, argo_n.LATITUDE.values, c=argo_n.N_PROF, transform=data_proj, cmap='cool')
plt.colorbar(map, label='Profile')

x_lims = [np.min(decimal_year), np.max(decimal_year)]
if x_lims[0] == x_lims[1]:
    x_lims[1] = x_lims[1]+10/365

plt.rcParams.update({'font.size': 14})  # Change 14 to the desired font size

for idx, var in enumerate(plot_vars):
    plot_exist = 0

    ax = f.add_subplot(gs[idx*2+2])
    if idx==0:
        plt.title(file + '_' + var)
    else:
        plt.title(var)

    if np.isnan(argo_n[var]).all():
        continue
    
    if var=='DOXY_ADJUSTED':
        color_map = 'plasma_r'
    elif var=='NITRATE_ADJUSTED':
        color_map = 'magma_r'
    elif var=='TEMP_ADJUSTED':
        color_map = 'cool'
    elif var=='TEMP_ADJUSTED':
        color_map = 'winter'
    elif var=='PH_IN_SITU_TOTAL_ADJUSTED':
        color_map = 'cividis'
    elif var=='PSAL_ADJUSTED':
        color_map = 'Wistia'
    elif var=='DIC':
        color_map = 'copper_r'
    elif var=='CHLA_ADJUSTED':
        color_map = 'spring'
    elif var=='BBP700_ADJUSTED':
        color_map = 'autumn'
    elif var=='CDOM_ADJUSTED':
        color_map = 'bone'
    else:
        color_map = 'viridis_r'

    # temp_press = np.zeros([len(argo_n.N_PROF), len(argo_n.N_LEVELS)])
    # temp_press[:] = np.nan
    # temp_var = np.zeros([len(argo_n.N_PROF), len(argo_n.N_LEVELS)])
    # temp_var[:] = np.nan
    # temp_date = np.zeros([len(argo_n.N_PROF), len(argo_n.N_LEVELS)])
    # temp_date[:] = np.nan


    #set color limits according to min/max of MLD properties
    if 'MLD' not in list_var:
        c_limit = [0, 1]
        y_limit = [100,0]
    elif np.logical_and(np.nansum(~np.isnan(argo_n.MLD))==0, ~np.isnan(argo_n[var].where(argo_n.PRES_ADJUSTED<150)).all()):
        c_limit = [np.nanmin(argo_n[var].where(argo_n.PRES_ADJUSTED<150)), np.nanmax(argo_n[var].where(argo_n.PRES_ADJUSTED<150))]
        y_limit = [200, 0]
    elif ~np.isnan(argo_n[var].where(argo_n.PRES_ADJUSTED<150)).all(): 
        c_limit = [np.nanmin(argo_n[var].where(argo_n.PRES_ADJUSTED<np.nanmax(argo_n.MLD))), np.nanmax(argo_n[var].where(argo_n.PRES_ADJUSTED<np.nanmax(argo_n.MLD)))]
        y_limit = [np.nanmax(argo_n.MLD)+50, 0]
    else:
        c_limit = [0, 1]
        y_limit = [100,0]


    for p in range(0, len(argo_n.N_PROF)):
        p_p = argo_n.PRES_ADJUSTED[p,np.logical_and(~np.isnan(argo_n[var][p,:]), ~np.isnan(argo_n.PRES_ADJUSTED[p,:]))].values

        t_p = argo_n.decimal_year[p:p+2].values
        if t_p.size==1:
            t_p = np.tile(t_p, (2,1))
            t_p[1] = t_p[1] + (argo_n.decimal_year[p] - argo_n.decimal_year[p-1]).values
        elif np.isnan(t_p).any(): # if any values in t_p are nans
            if np.isnan(t_p).all(): # if all are nans, continue
                continue
            elif np.isnan(t_p[0]):
                t_p[0] = t_p[1] - 10/365
            else:
                t_p[1] = t_p[0] + 10/365

        xl,yl = np.meshgrid(t_p, p_p)


        c = argo_n[var][p,np.logical_and(~np.isnan(argo_n[var][p,:]), ~np.isnan(argo_n.PRES_ADJUSTED[p,:]))].values
        if c.size==0:
            continue
        c = np.tile(c, (2,1))
        c = c.T
        plt.pcolormesh(xl, yl, c[0:-1,0:-1], cmap=color_map, shading='flat')
        plt.clim(c_limit)
        plot_exist = 1

    plt.ylim(y_limit)
    

    if 'MLD' in list_var:
        plt.plot(argo_n.decimal_year, argo_n.MLD, 'm')

    plt.xlim(x_lims)
    if plot_exist==1:
        plt.colorbar(label=var)

    ax = f.add_subplot(gs[idx*2+3])

    # axs1
    #set color limits according to the min / max water below 500m

    if ~np.isnan(argo_n['PRES_ADJUSTED'].where(argo_n.PRES_ADJUSTED>500)).all():
        c_limit = [np.nanmin(argo_n[var].where(argo_n.PRES_ADJUSTED>500)), np.nanmax(argo_n[var].where(argo_n.PRES_ADJUSTED>500))]

    for p in range(0, len(argo_n.N_PROF)):
        p_p = argo_n.PRES_ADJUSTED[p,np.logical_and(~np.isnan(argo_n[var][p,:]), ~np.isnan(argo_n.PRES_ADJUSTED[p,:]))].values

        t_p = argo_n.decimal_year[p:p+2].values
        if t_p.size==1:
            t_p = np.tile(t_p, (2,1))
            t_p[1] = t_p[1] + (argo_n.decimal_year[p] - argo_n.decimal_year[p-1]).values
        elif np.isnan(t_p).any(): # if any values in t_p are nans
            if np.isnan(t_p).all(): # if all are nans, continue
                continue
            elif np.isnan(t_p[0]):
                t_p[0] = t_p[1] - 10/365
            else:
                t_p[1] = t_p[0] + 10/365
        xl,yl = np.meshgrid(t_p, p_p)


        c = argo_n[var][p,np.logical_and(~np.isnan(argo_n[var][p,:]), ~np.isnan(argo_n.PRES_ADJUSTED[p,:]))].values
        if c.size==0:
            continue
        c = np.tile(c, (2,1))
        c = c.T
        plt.pcolormesh(xl, yl, c[0:-1,0:-1], cmap=color_map, shading='flat')    
        plt.clim(c_limit)
    if plot_exist==1:
        plt.colorbar(label=var)
    if 'MLD' in list_var:
        plt.plot(argo_n.decimal_year, argo_n.MLD, 'm')
    if ~np.isnan(argo_n['PRES_ADJUSTED']).all():
        plt.ylim([np.nanmax(argo_n.PRES_ADJUSTED), 0])    
    plt.xlim(x_lims)
plt.tight_layout()
plot_filename = file[0:-3]
plt.savefig(f'{processed_fig_dir}{plot_filename}_v2.png')
argo_n.close()        
